# Notebook 09 – Thử nghiệm mở rộng: bảng thuế và lịch sử hồ sơ vay độ sâu 2

Các nghiên cứu trên cùng bộ dữ liệu có dùng thêm **dữ liệu cơ quan thuế** (`tax_registry_a/b/c`) và **bảng độ sâu 2** (nhiều bản ghi trên mỗi hồ sơ cũ, đánh chỉ số bởi `num_group1`, `num_group2`). Notebook này kiểm tra, **có đối chứng**, việc thêm các bảng đó có cải thiện mô hình cuối không.

- Chỉ dùng cột thật của bảng, gộp về 1 dòng/hồ sơ bằng cùng hàm như notebook 02 (số bản ghi, max, mean, giá trị đầu); ngày đổi thành số ngày so với ngày quyết định.
- Áp dụng cùng các quy tắc chống rò rỉ: loại cột ngày phát sinh sau quyết định và cột trôi độ sẵn có (train vs valid, không dùng test).
- So sánh với mô hình cuối (notebook 05) trên cùng tập test bằng **kiểm định DeLong** và Gini theo tuần. Chỉ giữ thay đổi nếu cải thiện có ý nghĩa thống kê.
- `credit_bureau_a_2` không dùng vì rất lớn (hàng trăm triệu dòng), dễ vượt bộ nhớ khi nộp bài.

Notebook này **không** thay đổi file nộp bài (notebook 07 vẫn dùng mô hình cuối của notebook 05).

In [ ]:
# Giai phong bo nho con lai tu cac phan truoc (chay TRUOC khi doc du lieu cua phan nay)
import utils
utils.free_memory(globals())

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, polars as pl, joblib
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from utils import META
OUT_DIR, SEED = utils.get_out_dir(), 42
defs = utils.load_definitions()
EXTRA = ["tax_registry_a_1", "tax_registry_b_1", "tax_registry_c_1", "applprev_2"]

base = utils.load_base()
blocks = []
for t in EXTRA:
    try:
        utils.table_files(t)
    except FileNotFoundError as e:
        print("Bo qua:", e); continue
    f = utils.collect(utils.depth1_features(t, base))
    print(t, "->", f.shape)
    blocks.append(f)
ext = blocks[0]
for f in blocks[1:]:
    ext = ext.join(f, on="case_id", how="full", coalesce=True)
ext = utils.prune_features(ext, max_null=0.95)
print("Dac trung moi:", ext.width - 1)

In [ ]:
# Chong ro ri: cot ngay sau quyet dinh + troi do san co (train vs valid)
df2 = pl.read_parquet(OUT_DIR / "features_v2.parquet").select(["case_id", "WEEK_NUM"])
new = [c for c in ext.columns if c != "case_id"]
tmp = df2.join(ext, on="case_id", how="left")
fut = utils.future_date_columns(tmp, new, defs, min_share=0.01)
drop = fut.loc[fut["loai"], "cot"].tolist()
weeks = np.sort(tmp["WEEK_NUM"].unique()); n_w = len(weeks)
w_tr, w_va = weeks[: int(n_w * 0.70)].tolist(), weeks[int(n_w * 0.70): int(n_w * 0.85)].tolist()
n_tr = tmp.filter(pl.col("WEEK_NUM").is_in(w_tr)).select([pl.col(c).is_null().mean().alias(c) for c in new]).row(0)
n_va = tmp.filter(pl.col("WEEK_NUM").is_in(w_va)).select([pl.col(c).is_null().mean().alias(c) for c in new]).row(0)
drop += [c for c, a, b in zip(new, n_tr, n_va) if abs(b - a) >= 0.30]
ext = ext.drop(sorted(set(drop)))
new = [c for c in ext.columns if c != "case_id"]
print("Loai:", len(set(drop)), "| giu lai:", len(new))
del tmp, df2

In [ ]:
pack = joblib.load(OUT_DIR / "final_model.joblib"); feats = pack["features"]
df = pl.read_parquet(OUT_DIR / "features_v2.parquet").join(ext, on="case_id", how="left").to_pandas()
train, valid, test = utils.time_split(df)
P = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=200, subsample=0.8, subsample_freq=1,
         colsample_bytree=0.6, reg_lambda=5.0, random_state=SEED, n_jobs=-1, verbose=-1, metric="auc")
m = lgb.LGBMClassifier(**P).fit(train[feats + new], train["target"], eval_set=[(valid[feats + new], valid["target"])],
                                callbacks=[lgb.early_stopping(200, verbose=False)])
p_new = m.predict_proba(test[feats + new])[:, 1]
p_old = pack["model"].predict_proba(test[feats])[:, 1]
dl = utils.delong_test(test["target"], p_new, p_old)
res = pd.DataFrame({
    "mo_hinh": ["Mo hinh cuoi (notebook 05)", "+ bang thue + applprev_2"],
    "so_bien": [len(feats), len(feats) + len(new)],
    "AUC_test": [dl["AUC_2"], dl["AUC_1"]],
    "Gini_tuan_tb": [utils.gini_by_week(test.assign(score=p_old)).mean(), utils.gini_by_week(test.assign(score=p_new)).mean()],
    "stability": [utils.gini_stability(test.assign(score=p_old))["stability"], utils.gini_stability(test.assign(score=p_new))["stability"]],
}).set_index("mo_hinh")
res.to_csv(OUT_DIR / "extra_tables_experiment.csv"); display(res.round(4))
print("DeLong: chenh AUC = %.4f | z = %.2f | p = %.3g -> %s" % (dl["chenh"], dl["z"], dl["p_value"],
      "cai thien co y nghia thong ke" if dl["p_value"] < 0.05 and dl["chenh"] > 0 else "khong cai thien co y nghia"))
imp = pd.Series(m.booster_.feature_importance("gain"), index=feats + new)
print("Ty trong gain cua bien moi: %.1f%%" % (100 * imp[new].sum() / imp.sum()))
display(pd.DataFrame({"gain_%": 100 * imp[new] / imp.sum(), "mo_ta": utils.describe(new, defs)}).sort_values("gain_%", ascending=False).head(10).round(2))

## Nhận xét (tự viết)
- Thêm bảng thuế và lịch sử hồ sơ vay độ sâu 2 có cải thiện AUC / độ ổn định có ý nghĩa thống kê không: ...
- Nếu có: cân nhắc chi phí (thời gian, bộ nhớ, độ phức tạp vận hành) so với mức cải thiện; nếu không: ghi nhận là kết quả âm tính có giá trị.